In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
# --------------------------------------------------
# Directories
# --------------------------------------------------
DIR_DATA = Path("data")
DIR_METADATA = DIR_DATA / "0_metadata"
DIR_PROCESSED = DIR_DATA / "2_processed"
DIR_RESULTS = DIR_DATA / "3_results"
DIR_EVAL = DIR_DATA / "4_evaluation"

DIR_ICEYE = DIR_PROCESSED / "1_ICEYE"
DIR_NDVI = DIR_PROCESSED / "2_Sentinel2"
DIR_DEPMAP = DIR_PROCESSED / "3_Terrain" / "depmap"
DIR_ZSCORE = DIR_PROCESSED / "4_ICEYE_Zscore"
DIR_GRADCAM_CLS = DIR_RESULTS / "siamese-gradcam-classification" / "patches-gradcam"
DIR_GRADCAM_REG = DIR_RESULTS / "siamese-gradcam-regression" / "patches-gradcam"

# --------------------------------------------------
# Files
# --------------------------------------------------
FILEPATH_PAIR_MANIFEST = DIR_METADATA / "pair-manifest.csv"
FILEPATH_EVAL_SUMMARY = DIR_EVAL / "correlation_summary.csv"
FILEPATH_PATCH_CENTERS = DIR_METADATA / "patch-centers.csv"


TARGET_PAIRS = ["DE01", ]

CORRELATION_METHOD = "spearman"

In [ ]:
def find_gradcam_file(directory, pair_id, patch_id):
    pattern = f"gradcam_{pair_id}_{patch_id}_*.tif"
    matches = sorted(directory.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"{pattern}: expected 1 file, found {len(matches)}"
        )
    return matches[0]

In [ ]:
# --------------------------------------------------
# Read manifest
# --------------------------------------------------
manifest = pd.read_csv(FILEPATH_PAIR_MANIFEST)

# Only DE01 for now
manifest = manifest[manifest["pair_id"].isin(TARGET_PAIRS)]

rows = []
for _, pair in manifest.iterrows():
    pair_id = pair["pair_id"]
    sar_target_dir = DIR_ICEYE / pair["sar_target"]

    gradcam_cls_dir = DIR_GRADCAM_CLS
    gradcam_reg_dir = DIR_GRADCAM_REG

    # ----------------------------------------------
    # Enumerate ICEYE test patches
    # ----------------------------------------------
    for sar_target_file in sorted(sar_target_dir.glob("*.tif")):

        stem = sar_target_file.stem

        # Example:
        # 4155938_EB24CC0120

        image_id, patch_id = stem.split("_", 1)

        # Test patches begin with E
        if not patch_id.startswith("E"):
            continue

        gradcam_cls_file = find_gradcam_file(
            gradcam_cls_dir,
            pair_id,
            patch_id,
        )
        gradcam_reg_file = find_gradcam_file(
            gradcam_reg_dir,
            pair_id,
            patch_id,
        )
        rows.append(
            {
                "pair_id": pair_id,
                "patch_id": patch_id,
                "gradcam_cls": gradcam_cls_file,
                "gradcam_reg": gradcam_reg_file,
            }
        )

groups_patch_paths = pd.DataFrame(rows)

print(groups_patch_paths.head())
print(f"\n{len(groups_patch_paths)} test patches found.")

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
results = []
for _, row in groups_patch_paths.iterrows():
    # ------------------------------
    # Compute correlations
    # ------------------------------
    results.append({
        "pair_id": row["pair_id"],
        "patch_id": row["patch_id"],
        "cls_prob": float(
            row["gradcam_cls"].stem.split("_prob")[-1]
        ),
        "reg_pred": float(
            row["gradcam_reg"].stem.split("_pred")[-1]
        ),
    })

eval_summary = pd.DataFrame(results)
eval_summary.head()

In [ ]:
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

In [ ]:
# ---------------------------------------
# Prepare data
# ---------------------------------------
plot_data = eval_summary[
    [
        "patch_id",
        "cls_prob",
        "reg_pred",
    ]
].dropna().copy()


# ---------------------------------------
# Spearman correlation
# ---------------------------------------
rho, p_value = spearmanr(
    plot_data["cls_prob"],
    plot_data["reg_pred"],
)

print(f"Spearman rho: {rho:.3f}")
print(f"p-value     : {p_value:.4f}")


# ---------------------------------------
# Plot
# ---------------------------------------
fig, ax = plt.subplots(
    figsize=(7, 6)
)

ax.scatter(
    plot_data["cls_prob"],
    plot_data["reg_pred"],
    color="gray",
    s=50,
    alpha=0.7,
)


# ---------------------------------------
# Patch IDs
# ---------------------------------------
for _, row in plot_data.iterrows():
    ax.text(
        row["cls_prob"],
        row["reg_pred"],
        row["patch_id"],
        fontsize=7,
    )


# ---------------------------------------
# Labels
# ---------------------------------------
ax.set_xlabel(
    "Classification probability"
)

ax.set_ylabel(
    "Regression prediction"
)

ax.set_xlim(
    0,
    1,
)

ax.grid(
    alpha=0.3,
)

ax.set_title(
    f"Classification vs. Regression Output\n"
    f"Spearman $\\rho$ = {rho:.3f}, "
    f"$p$ = {p_value:.3g}"
)

plt.tight_layout()


# ---------------------------------------
# Save
# ---------------------------------------
output_path = (
    DIR_EVAL /
    "fig_siamese_classification_vs_regression_dry.png"
)

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

print(f"Saved: {output_path}")


# ---------------------------------------
# Display
# ---------------------------------------
plt.show()
plt.close(fig)